In [1]:
%pip install -U pandas pyarrow numpy requests tqdm torch transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json, re, time, hashlib, unicodedata
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
from IPython.display import display

# ===== SETTINGS YOU CAN CHANGE =====
DATA_DIR = Path("pmc/processed")
SAVE_DIR = Path("pmc/medcpt_llm_pilot")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = SAVE_DIR / "raw_llm"
RAW_DIR.mkdir(exist_ok=True)

PILOT_ARTICLES = 3        # 3 for first test, None for all articles
MIN_PASSAGE_WORDS = 12
CHUNK_TARGET_WORDS = 145
CHUNK_OVERLAP_WORDS = 18
MEDCPT_MAX_TOKENS = 512
BATCH_SIZE = 12
TOP_PER_QUERY = 3
MAX_CHUNKS_PER_ARTICLE = 12

DOC_MODEL_NAME = "ncbi/MedCPT-Article-Encoder"
QUERY_MODEL_NAME = "ncbi/MedCPT-Query-Encoder"
LLM_MODEL = "qwen2.5:7b-instruct"
OLLAMA_BASE_URL = "http://localhost:11434"
LLM_CONTEXT_WINDOW = 8192
SEED = 42

# Do not change query wording after examining results without recording the experiment.
RETRIEVAL_QUERIES = {
    "case_definition": "How did investigators define or diagnose autoimmune gastritis in study participants? inclusion criteria reference standard",
    "histology": "autoimmune gastritis corpus-predominant atrophic gastritis biopsy histological diagnostic criteria",
    "autoantibodies": "antiparietal cell antibodies intrinsic factor antibodies diagnostic criteria autoimmune gastritis",
    "biomarkers": "gastrin pepsinogen I pepsinogen ratio vitamin B12 diagnostic threshold autoimmune gastritis",
    "endoscopy": "endoscopy gastric corpus atrophy diagnostic evidence criteria autoimmune gastritis",
    "pernicious_anaemia": "pernicious anemia pernicious anaemia diagnostic criteria intrinsic factor antibody B12 deficiency",
}
print("Output folder:", SAVE_DIR.resolve())

Output folder: C:\Users\caral\OneDrive\Documents\SCI501\SCI501_Project\pmc\medcpt_llm_pilot


In [3]:
article_path = DATA_DIR / "articles.parquet"
passage_path = DATA_DIR / "passages.parquet"
if not article_path.exists() or not passage_path.exists():
    raise FileNotFoundError("Run data_pipeline(2).ipynb first; expected pmc/processed/*.parquet")
articles = pd.read_parquet(article_path).drop_duplicates(subset="pmcid").copy()
passages = pd.read_parquet(passage_path).copy()
required_articles = {"pmcid", "title", "year"}
required_passages = {"pmcid", "passage_id", "section", "text"}
assert required_articles.issubset(articles), f"Missing article columns: {required_articles-set(articles)}"
assert required_passages.issubset(passages), f"Missing passage columns: {required_passages-set(passages)}"
articles["pmcid"] = articles["pmcid"].astype(str)
passages["pmcid"] = passages["pmcid"].astype(str)
articles["year"] = pd.to_numeric(articles["year"], errors="coerce").astype("Int64")

# Deterministic pilot: earliest PMCID sort, *not* a representative historical sample.
chosen_ids = articles["pmcid"].sort_values().tolist()
if PILOT_ARTICLES is not None:
    chosen_ids = chosen_ids[:PILOT_ARTICLES]
selected_articles = articles.loc[articles["pmcid"].isin(chosen_ids)].copy()
excluded_sections = r"REFEREN|BIBLIOGRAPH|ACKNOWLEDG|FUNDING|COMPETING|CONFLICT|AUTHOR.CONTRIB|SUPPLEMENT"
pool = passages.loc[passages["pmcid"].isin(chosen_ids)].copy()
pool = pool.loc[~pool["section"].fillna("").astype(str).str.contains(excluded_sections, case=False, regex=True)]
if "type" in pool.columns:
    pool = pool.loc[~pool["type"].fillna("").astype(str).isin(["title_1", "title_2"])]
pool["text"] = pool["text"].fillna("").astype(str).str.strip()
pool = pool.loc[pool["text"].str.split().str.len() >= MIN_PASSAGE_WORDS]
pool = pool.sort_values(["pmcid", "passage_id"]).reset_index(drop=True)
if pool.empty:
    raise ValueError("No eligible passages after filtering")
print("Articles available:", articles["pmcid"].nunique())
print("Pilot articles:", selected_articles["pmcid"].nunique())
print("Eligible BioC passages:", len(pool))
display(selected_articles[[c for c in ["pmcid", "year", "title"] if c in selected_articles]].head())

Articles available: 50
Pilot articles: 3
Eligible BioC passages: 259


,pmcid,year,title
0,PMC10311608,2023,Paradoxical association between dyspepsia and ...
1,PMC10456958,2023,Gastric Microbiota Gender Differences in Subje...
2,PMC10496934,2023,A Comprehensive Review of Upper Gastrointestin...


In [4]:
try:
    check = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=8)
    check.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        "Ollama is not running or is unreachable. Install/start Ollama, then run "
        f"'ollama pull {LLM_MODEL}' in a terminal."
    ) from exc
installed_models = [entry.get("name", "") for entry in check.json().get("models", [])]
print("Locally installed Ollama models:", installed_models)
if LLM_MODEL not in installed_models:
    raise RuntimeError(f"Model {LLM_MODEL!r} is not installed. Run: ollama pull {LLM_MODEL}")

RuntimeError: Ollama is not running or is unreachable. Install/start Ollama, then run 'ollama pull qwen2.5:7b-instruct' in a terminal.

In [5]:
import torch
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MedCPT encoder device:", device)
doc_tok = AutoTokenizer.from_pretrained(DOC_MODEL_NAME)
doc_encoder = AutoModel.from_pretrained(DOC_MODEL_NAME).to(device).eval()
query_tok = AutoTokenizer.from_pretrained(QUERY_MODEL_NAME)
query_encoder = AutoModel.from_pretrained(QUERY_MODEL_NAME).to(device).eval()

@torch.inference_mode()
def encode_queries(queries):
    encoded = query_tok(queries, padding=True, truncation=True, max_length=64, return_tensors="pt")
    outputs = query_encoder(**{k: v.to(device) for k, v in encoded.items()})
    return outputs.last_hidden_state[:, 0, :].cpu().numpy().astype("float32")

@torch.inference_mode()
def encode_documents(pairs, batch_size=BATCH_SIZE):
    vectors = []
    for i in tqdm(range(0, len(pairs), batch_size), desc="MedCPT passage encoding"):
        batch = pairs[i:i+batch_size]
        enc = doc_tok([p[0] for p in batch], [p[1] for p in batch],
                      padding=True, truncation=False, return_tensors="pt")
        if enc["input_ids"].shape[1] > MEDCPT_MAX_TOKENS:
            raise ValueError("Encoder input exceeds 512 tokens: adjust chunking, do not silently truncate")
        out = doc_encoder(**{k: v.to(device) for k, v in enc.items()})
        vectors.append(out.last_hidden_state[:, 0, :].cpu().numpy().astype("float32"))
    return np.vstack(vectors)

def unit_rows(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

c:\Users\caral\OneDrive\Documents\SCI501\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MedCPT encoder device: cpu


c:\Users\caral\OneDrive\Documents\SCI501\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\caral\.cache\huggingface\hub\models--ncbi--MedCPT-Article-Encoder. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\caral\OneDrive\Documents\SCI501\.venv\Lib\site-packages\huggingface_hub\file_download